# Reading in a short story as text sample into Python

In [73]:
with open("data/a-few-good-men.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    print("Total numer of characters: ", len(raw_text))
    print(raw_text[:99])

Total numer of characters:  327004
 A FEW GOOD MEN



                                        Written by

                            


# Tokenizing Text

In [89]:
import re

preprocessed = re.findall(r"\w+|[^\w\s]", raw_text)
print("Number of Tokens: ", len(preprocessed))
print(preprocessed[:20])

Number of Tokens:  35834
['A', 'FEW', 'GOOD', 'MEN', 'Written', 'by', 'Aaron', 'Sorkin', 'Revised', 'Third', 'Draft', 'July', '15', ',', '1991', 'FADE', 'IN', ':', 'EXT', '.']


# Converting tokens into token IDs

In [75]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print("Vocabulary size: ", vocab_size)

Vocabulary size:  3714


## Creating a vocabulary

In [91]:
vocab = {token : i for i, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 20:
        break

('!', 0)
('"', 1)
('#', 2)
('&', 3)
("'", 4)
('(', 5)
(')', 6)
(',', 7)
('-', 8)
('.', 9)
('/', 10)
('00', 11)
('0600', 12)
('1', 13)
('1000', 14)
('14', 15)
('14th', 16)
('15', 17)
('16', 18)
('17', 19)
('18th', 20)


# Simple text tokinizer

In [77]:
class SimpleTokenizer:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items() }

    def encode(self, text):
        preprocessed = re.findall(r"\w+|[^\w\s]", text)
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids):
        return " ". join([self.int_to_str[i] for i in ids])

tokenizer = SimpleTokenizer(vocab)
text = """Please the Court, is this dialogue
        relevant to anything in particular?"""

print("Text: ", text)
ids = tokenizer.encode(text)
print("\nEncoded text: ", ids)

print("\nDecoded text: ", tokenizer.decode(ids))

Text:  Please the Court, is this dialogue
        relevant to anything in particular?

Encoded text:  [695, 3415, 251, 7, 2195, 3431, 1598, 2914, 3453, 1078, 2142, 2661, 59]

Decoded text:  Please the Court , is this dialogue relevant to anything in particular ?


## Adding special context token

In [78]:
print("\nSize of vocab befor special context token: ", len(vocab))
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token : i for i, token in enumerate(all_tokens)}
print("\nSize of vocab after special context token: ", len(vocab))

print("\nlast 5 elements of vocab: ")
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)


Size of vocab befor special context token:  3714

Size of vocab after special context token:  3716

last 5 elements of vocab: 
('yourself', 3711)
('zero', 3712)
('zippity', 3713)
('<|endoftext|>', 3714)
('<|unk|>', 3715)


## Tokenizer that handles unknown words

In [85]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items() }

    def encode(self, text):
        preprocessed = re.findall(r"\w+|[^\w\s]", text)
        preprocessed = [s if s in self.str_to_int else "<|unk|>" for s in preprocessed]
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids):
        decode_text = " ". join([self.int_to_str[i] for i in ids])
        return re.sub(r"\s+([,.?!])", r"\1", decode_text)

tokenizer = SimpleTokenizerV2(vocab)
text1 = "Did you wear that uniform on the plane? The astronaut smiled."
text2 = "Did you wear that uniform on the plane? Quantum drift begins."
text = " <|endoftext|> ".join((text1, text2))

print("Text: ", text)
ids = tokenizer.encode(text)
print("\nEncoded text: ", ids)

print("\nDecoded text: ", tokenizer.decode(ids))




Text:  Did you wear that uniform on the plane? The astronaut smiled. <|endoftext|> Did you wear that uniform on the plane? Quantum drift begins.

Encoded text:  [281, 3707, 3613, 3414, 3538, 2591, 3415, 2724, 59, 876, 3715, 3715, 9, 3715, 3715, 3715, 3715, 3715, 281, 3707, 3613, 3414, 3538, 2591, 3415, 2724, 59, 3715, 3715, 1206, 9]

Decoded text:  Did you wear that uniform on the plane? The <|unk|> <|unk|>. <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> Did you wear that uniform on the plane? <|unk|> <|unk|> begins.
